In [3]:
import pandas as pd
import numpy as np
import os
import ast
import datetime
import torch


MAX_SEQ_LENGTH = 20
MIN_INTERACTIONS = 5


In [8]:
print("Loading data...")
# Assuming 'ratings.csv' is in your current directory or path
df = pd.read_csv('data/ml-25m/ratings.csv')

# Rename columns to standard names
df = df.rename(columns={'userId': 'user_id', 'movieId': 'item_id'})

print(f"Loaded {len(df)} total interactions.")
display(df.head())

Loading data...
Loaded 25000095 total interactions.


,user_id,item_id,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [9]:
# Sort chronologically
df = df.sort_values(by=['user_id', 'timestamp'])

# Filter out users with fewer than MIN_INTERACTIONS
user_counts = df['user_id'].value_counts()
valid_users = user_counts[user_counts >= MIN_INTERACTIONS].index
df = df[df['user_id'].isin(valid_users)].copy()

# Create contiguous integer mappings (1-indexed)
user_ids = df['user_id'].unique()
item_ids = df['item_id'].unique()

user2idx = {u: i + 1 for i, u in enumerate(user_ids)}
item2idx = {i: j + 1 for j, i in enumerate(item_ids)}

df['user_idx'] = df['user_id'].map(user2idx)
df['item_idx'] = df['item_id'].map(item2idx)

print(f"Filtered Data: {len(df)} interactions.")
print(f"Unique Users: {len(user_ids)} | Unique Items: {len(item_ids)}")


Filtered Data: 25000095 interactions.
Unique Users: 162541 | Unique Items: 59047


In [10]:

print("Grouping into chronological sequences and applying padding...")

# Group interactions into lists per user
user_seqs = df.groupby('user_idx')['item_idx'].apply(list).to_dict()

train_X, train_y = [], []
val_X, val_y = [], []
test_X, test_y = [], []

for u, seq in user_seqs.items():
    # Iterate through the user's sequence to create targets
    for i in range(1, len(seq)):
        target = seq[i]
        
        # Get up to MAX_SEQ_LENGTH previous items as context
        context = seq[max(0, i - MAX_SEQ_LENGTH):i]
        
        # Native Pre-padding: Pad context with 0s at the beginning
        pad_len = MAX_SEQ_LENGTH - len(context)
        padded_context = [0] * pad_len + context
        
        # Leave-one-out split strategy per user
        if i == len(seq) - 1:
            test_X.append(padded_context)
            test_y.append(target)
        elif i == len(seq) - 2:
            val_X.append(padded_context)
            val_y.append(target)
        else:
            train_X.append(padded_context)
            train_y.append(target)

print(f"Train samples: {len(train_X)}")
print(f"Val samples:   {len(val_X)}")
print(f"Test samples:  {len(test_X)}")


Grouping into chronological sequences and applying padding...
Train samples: 24512472
Val samples:   162541
Test samples:  162541


In [11]:
print("Converting to PyTorch Tensors...")

# Convert to LongTensors for PyTorch nn.Embedding layers
train_data = {
    'X': torch.tensor(train_X, dtype=torch.long), 
    'y': torch.tensor(train_y, dtype=torch.long)
}
val_data = {
    'X': torch.tensor(val_X, dtype=torch.long), 
    'y': torch.tensor(val_y, dtype=torch.long)
}
test_data = {
    'X': torch.tensor(test_X, dtype=torch.long), 
    'y': torch.tensor(test_y, dtype=torch.long)
}

# data dictionaries
torch.save(train_data, 'train_data.pt')
torch.save(val_data, 'val_data.pt')
torch.save(test_data, 'test_data.pt')

# Save mapping dictionaries 
import pickle
with open('mappings.pkl', 'wb') as f:
    pickle.dump({'user2idx': user2idx, 'item2idx': item2idx}, f)

print("Data successfully prepared and saved!")


Converting to PyTorch Tensors...
Data successfully prepared and saved!
